In [ ]:
!pip install pandas --break-system-packages
!pip install matplotlib --break-system-packages

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

T_step = 0.001

df = pd.read_csv("output.csv") if os.path.exists("output.csv") else None
cpu_df = pd.read_csv("output_cpu.csv") if os.path.exists("output_cpu.csv") else None
cuda_df = pd.read_csv("output_cuda.csv") if os.path.exists("output_cuda.csv") else None

if df is not None:
    df['time'] = df['step'] * T_step
if cpu_df is not None:
    cpu_df['time'] = cpu_df['step'] * T_step
if cuda_df is not None:
    cuda_df['time'] = cuda_df['step'] * T_step

In [ ]:
if df is not None:
    fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)

    axes[0].plot(df['time'], df['input'], label='Input Current I(t)', color='green')
    axes[0].axvspan(2.0, 3.0, color='red', alpha=0.1, label='Hyperpolarizing block')
    axes[0].axvspan(5.0, 8.0, color='blue', alpha=0.1, label='Drive')
    axes[0].set_ylabel('Current')
    axes[0].legend()
    axes[0].set_title('Input Stimulus')

    axes[1].plot(df['time'], df['v'], label='Membrane Potential (v)', color='blue')
    axes[1].plot(df['time'], df['v_th_eff'], '--', label='Effective Threshold (v_th + w)', color='red')
    axes[1].set_ylabel('Potential')
    axes[1].legend()
    axes[1].set_title('ALIF Neuron Dynamics')

    axes[2].vlines(df[df['spike']]['time'], 0, 1, color='black', alpha=0.8, label='Spikes')
    axes[2].plot(df['time'], df['synapse'], label='Synaptic Conductance', color='orange')
    axes[2].set_ylabel('Spike / Conductance')
    axes[2].legend()
    axes[2].set_title('Outputs')

    axes[3].plot(df['time'], df['w'], label='Adaptation w', color='purple')
    axes[3].set_xlabel('Time (s)')
    axes[3].set_ylabel('w')
    axes[3].legend()
    axes[3].set_title('Threshold Adaptation')

    plt.tight_layout()
    plt.show()
else:
    print("Skipping Figure 1: no data")

In [ ]:
if df is not None:
    spikes = df[df['spike']]
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.stem(spikes['time'], [1]*len(spikes), basefmt=' ', linefmt='k|', markerfmt='k_')
    ax.set_xlabel('Time (s)')
    ax.set_yticks([])
    ax.set_title('Spike Raster')
    plt.tight_layout()
    plt.show()
else:
    print("Skipping Figure 2: no data")

In [ ]:
if cpu_df is not None and cuda_df is not None:
    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    axes[0].plot(cpu_df['time'], cpu_df['v'], label='CPU v', color='blue', alpha=0.7)
    axes[0].plot(cuda_df['time'], cuda_df['v'], label='CUDA v', color='orange', alpha=0.7)
    axes[0].set_ylabel('Membrane Potential')
    axes[0].legend()
    axes[0].set_title('Backend Comparison: v(t)')

    diff = cpu_df['v'] - cuda_df['v']
    axes[1].plot(cpu_df['time'], diff, label='CPU - CUDA', color='red')
    axes[1].axhline(0, color='black', linestyle='--')
    axes[1].set_xlabel('Time (s)')
    axes[1].set_ylabel('Difference')
    axes[1].legend()
    axes[1].set_title('Difference (should be near zero)')
    plt.tight_layout()
    plt.show()
else:
    print("Skipping Figure 3: need both output_cpu.csv and output_cuda.csv")

In [ ]:
if df is not None:
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.plot(df['v'], df['w'], color='purple', alpha=0.6, linewidth=0.5)
    
    spikes = df[df['spike']]
    ax.scatter(spikes['v'], spikes['w'], color='black', s=10, label='Spike events')
    
    ax.set_xlabel('Membrane Potential (v)')
    ax.set_ylabel('Adaptation (w)')
    ax.set_title('ALIF State Space: w vs v')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Skipping Figure 4: no data")

In [ ]:
if df is not None:
    zoom = df[(df['time'] >= 1.9) & (df['time'] <= 3.1)]
    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    
    axes[0].plot(zoom['time'], zoom['input'], label='Input I(t)', color='green')
    axes[0].axvline(2.0, color='red', linestyle='--', alpha=0.5, label='Step onset')
    axes[0].set_ylabel('Current')
    axes[0].legend()
    axes[0].set_title('Synapse Filtering: Step Response')
    
    axes[1].plot(zoom['time'], zoom['synapse'], label='Conductance g(t)', color='orange')
    axes[1].axvline(2.0, color='red', linestyle='--', alpha=0.5)
    axes[1].axvline(3.0, color='red', linestyle='--', alpha=0.5, label='Step offset')
    axes[1].set_xlabel('Time (s)')
    axes[1].set_ylabel('Conductance')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("Skipping Figure 5: no data")